In [15]:
# Description: This script trains an AST whose input has been modified to take audio insteasd of patches of images 
# Original code is based off a tutorial by Brian Pulfer
# https://medium.com/@brianpulfer/vision-transformers-from-scratch-pytorch-a-step-by-step-guide-96c3313c2e0c
# Andrei Cartera -- Mar 2025

import datetime
import numpy as np
import CustomSpeechCommands_Repcycle as SpeechCommands
from AudioTransformer import AudioTransformer 
from tqdm.notebook import tqdm, trange
from pathlib import Path
import torch
import torch.nn as nn
from torch.optim import Adam, lr_scheduler
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(torch.__version__)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")


2.8.0+cu128
PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Device name: NVIDIA GeForce GTX 1080 Ti


In [16]:
classes = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
NUM_CLASSES = 10

# Hyperparameters
N_SEGMENTS = 32
REPC_VEC_SIZE = 64

EPOCHS = 50 #50
N_HEADS = 8
N_ENCODERS = 4
BATCH_SIZE = 64 #64
HIDDEN_DIM = 32
DROPOUT = 0.15
ACTIVATION="gelu"
LR = 0.0009

today = datetime.date.today()

MODEL_PATH = f'models/({today})FAST_FFT_NOISY_model_{N_SEGMENTS}SEG_{REPC_VEC_SIZE}VEC_E{EPOCHS}_{N_HEADS}_{N_ENCODERS}_B{BATCH_SIZE}_H{HIDDEN_DIM}.pth'

print(f"Model path: {MODEL_PATH}")


Model path: models/(2025-08-27)FAST_FFT_NOISY_model_32SEG_64VEC_E50_8_4_B64_H32.pth


In [17]:
def train():
  # Loading data
  
  print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")
  model = AudioTransformer(N_SEGMENTS, REPC_VEC_SIZE, N_ENCODERS, HIDDEN_DIM, N_HEADS, NUM_CLASSES).to(device)

  train_set = SpeechCommands.CustomSpeechCommandsDataset_Repcycle("../datasets/custom_speech_commands", n_segments=N_SEGMENTS, shuffle=False, vec_size=REPC_VEC_SIZE)
  #train_loader = DataLoader(train_set, shuffle=True, batch_size=BATCH_SIZE, drop_last=True)
  train_loader = DataLoader(train_set, shuffle=True, batch_size=BATCH_SIZE, num_workers=10, pin_memory=True, persistent_workers=True, drop_last=True)

  # Defining model and training options

  # Training loop
  optimizer = Adam(model.parameters(), lr=LR)
  scheduler = lr_scheduler.LinearLR(optimizer)
  criterion = CrossEntropyLoss()

  model.train()  # Set the model to training mode                                     
  for epoch in trange(EPOCHS, desc="Training"):
    train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} in training", leave=False):
      x, y = batch
      x, y = x.to(device), y.to(device)
      y_hat = model(x)
      loss = criterion(y_hat, y)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_loss += loss.item() * x.size(0)
      
    train_loss /= len(train_loader.dataset) 
    scheduler.step(train_loss)
    torch.cuda.empty_cache()
        
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{EPOCHS} loss: {train_loss:.2f}, LR: {current_lr}")
  
  torch.save(model.state_dict(), MODEL_PATH)
  print(f"Model saved as {MODEL_PATH}")



In [18]:
def test():
  model = AudioTransformer(N_SEGMENTS, REPC_VEC_SIZE, N_ENCODERS, HIDDEN_DIM, N_HEADS, NUM_CLASSES).to(device)
  model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
  #models/(2025-07-29)ATmodel_32SEG_64VEC_E50_8_4_B64_H32.pth
  print(f"Model loaded from {MODEL_PATH}")
  
  model.eval()  # Set the model to evaluation mode
  
  test_set = SpeechCommands.CustomSpeechCommandsDataset_Repcycle("../datasets/custom_speech_commands", n_segments=N_SEGMENTS, subset="testing", shuffle=True, vec_size=REPC_VEC_SIZE)
  test_loader = DataLoader(test_set, shuffle=True, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, persistent_workers=True, drop_last=True)

  criterion = CrossEntropyLoss()

  # Test loop
  with torch.no_grad():
    correct, total = 0, 0
    test_loss = 0.0
    for batch in tqdm(test_loader, desc="Testing"):
      x, y = batch

      x, y = x.to(device), y.to(device)
      y_hat = model(x)
      loss = criterion(y_hat, y)
      test_loss += loss.detach().cpu().item() / len(test_loader)

      correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
      total += len(x) 
      
    print(f"Test loss: {test_loss:.2f}")
    print(f"Test accuracy: {correct / total * 100:.2f}%")

In [19]:
if __name__ == "__main__":
  train()
  test() 

Using device:  cuda:0 (NVIDIA GeForce GTX 1080 Ti)


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1 in training:   0%|          | 0/543 [00:00<?, ?it/s]

Epoch 1/50 loss: 2.28, LR: 0.0005732772413682213


c:\Users\Andrew\Documents\GitHub\Audio-Transformer\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


...

Epoch 49/50 loss: 1.77, LR: 0.0005121426002162855


Epoch 50 in training:   0%|          | 0/543 [00:00<?, ?it/s]

Epoch 50/50 loss: 1.77, LR: 0.0005123934347410798
Model saved as models/(2025-08-27)FAST_FFT_NOISY_model_32SEG_64VEC_E50_8_4_B64_H32.pth
Model loaded from models/(2025-08-27)FAST_FFT_NOISY_model_32SEG_64VEC_E50_8_4_B64_H32.pth


Testing:   0%|          | 0/64 [00:00<?, ?it/s]

Test loss: 1.81
Test accuracy: 65.19%
